# Student starter
Run cells in order. An incomplete TODO prints guidance and stops subsequent cells. After implementing it, restart the kernel and run again.

# Exercise 7 — Semantic Search

Generalize pairwise sentence similarity to query versus every description. Reuse the exact CPU MiniLM model, manual cosine and stable ranking. Model setup/cache are starter code. Preload explicitly before running; the notebook never downloads. No independent algorithms or vector stores.

In [ ]:
if not globals().get("_workshop_todo_incomplete", False):
    try:
        from pathlib import Path
        import sys
        root = Path.cwd()
        if root.name == "notebooks":
            root = root.parent
        if not (root / "src").exists():
            raise RuntimeError("Start in repository root or notebooks directory.")
        sys.path.insert(0,str(root))
        from src.sentence_resources import load_sentence_embedding_model, SENTENCE_MODEL_ID, SENTENCE_MODEL_REVISION
        from src.retrieval import (encode_corpus_documents, encode_search_query, calculate_semantic_similarities,
                                  retrieve_semantic_results, semantic_search, prepare_search_corpus, search_tfidf)
        from workshop.course_data import load_course_documents, DEFAULT_QUERIES
        model = load_sentence_embedding_model()
        documents = load_course_documents()
        print(SENTENCE_MODEL_ID, SENTENCE_MODEL_REVISION, model.device)
        print(len(documents), documents[0])

    except NotImplementedError as error:
        print(str(error))
        print("Implement the indicated TODO, restart the kernel, and run again.")
        _workshop_todo_incomplete = True
else:
    print("Earlier TODO incomplete; later cells are paused.")


## Corpus → matrix

Only descriptions are encoded/indexed, never labels/keywords. Keep metadata and matrix rows aligned. Source functions reuse Exercise 6 encoding; N documents give N×384. Preserve the model-native Normalize layer. No extra normalization is requested.

In [ ]:
if not globals().get("_workshop_todo_incomplete", False):
    try:
        document_embeddings = encode_corpus_documents(model, documents)
        print(document_embeddings.shape)
        assert document_embeddings.shape == (20,384)
        print("First row preview:", document_embeddings[0,:10])
        lexical_corpus = prepare_search_corpus(documents)

    except NotImplementedError as error:
        print(str(error))
        print("Implement the indicated TODO, restart the kernel, and run again.")
        _workshop_todo_incomplete = True
else:
    print("Earlier TODO incomplete; later cells are paused.")


## Query → same space → cosine against every row

One query has 384 dimensions. One score per original document; sorting comes afterwards. Blank queries are rejected; zero-vector safety comes from the existing manual cosine.

In [ ]:
if not globals().get("_workshop_todo_incomplete", False):
    try:
        query = "Which subject teaches machines to work with human communication?"
        query_vector = encode_search_query(model,query)
        scores = calculate_semantic_similarities(query_vector,document_embeddings)
        print("Query shape:", len(query_vector), "Preview:", query_vector[:10])
        for position,(document,score) in enumerate(zip(documents,scores)):
            print(position,document["title"],round(score,6))

    except NotImplementedError as error:
        print(str(error))
        print("Implement the indicated TODO, restart the kernel, and run again.")
        _workshop_todo_incomplete = True
else:
    print("Earlier TODO incomplete; later cells are paused.")


## Scores → descending ranking → Top-K

Reuse Exercise 4 stable ranking and metadata dictionaries. Ties retain original order. Default K=5; no corpus mutation.

In [ ]:
if not globals().get("_workshop_todo_incomplete", False):
    try:
        top_k = 5
        results = retrieve_semantic_results(documents,scores,top_k)
        semantic = semantic_search(query,documents,document_embeddings,model,top_k)
        assert semantic["results"] == results
        print("Sorted indices:", semantic["sorted_indices"])
        for rank,result in enumerate(results,1):
            print(rank,result["title"],round(result["score"],6),result["text"])

    except NotImplementedError as error:
        print(str(error))
        print("Implement the indicated TODO, restart the kernel, and run again.")
        _workshop_todo_incomplete = True
else:
    print("Earlier TODO incomplete; later cells are paused.")


## Same query, same descriptions, same K — lexical vs semantic

Compare ranks/relevance, not score magnitudes as if identically calibrated. Keep description-only Task 06 IDF and preprocessing. Full corpus ranking now tests what Task 08 two-course comparison could not establish.

In [ ]:
if not globals().get("_workshop_todo_incomplete", False):
    try:
        lexical = search_tfidf(query,lexical_corpus,top_k)
        comparison = [{"Rank":i+1,"TF-IDF":lexical["results"][i]["title"],"TF-IDF cosine":lexical["results"][i]["score"],
                       "Semantic":semantic["results"][i]["title"],"Semantic cosine":semantic["results"][i]["score"]}
                      for i in range(top_k)]
        for row in comparison:
            print(row)
        for text in DEFAULT_QUERIES:
            classical = search_tfidf(text,lexical_corpus,top_k)["results"]
            dense = semantic_search(text,documents,document_embeddings,model,top_k)["results"]
            print(text,"TF-IDF:",classical[0]["title"],"Semantic:",dense[0]["title"])

    except NotImplementedError as error:
        print(str(error))
        print("Implement the indicated TODO, restart the kernel, and run again.")
        _workshop_todo_incomplete = True
else:
    print("Earlier TODO incomplete; later cells are paused.")


## Ambiguous/unrelated queries

Predict before running. Similarity is not probability or a fact check. For unrelated text, semantic search still returns a nearest course; all-zero TF-IDF returns original-order ties. Neither result establishes relevance. No threshold/abstention or generation is introduced.

In [ ]:
if not globals().get("_workshop_todo_incomplete", False):
    try:
        for text in ["learning","How do I bake a chocolate cake?"]:
            print(text)
            print("Lexical:",[(r["title"],round(r["score"],6)) for r in search_tfidf(text,lexical_corpus,3)["results"]])
            print("Semantic:",[(r["title"],round(r["score"],6)) for r in semantic_search(text,documents,document_embeddings,model,3)["results"]])

    except NotImplementedError as error:
        print(str(error))
        print("Implement the indicated TODO, restart the kernel, and run again.")
        _workshop_todo_incomplete = True
else:
    print("Earlier TODO incomplete; later cells are paused.")


## Instructor benchmark experiment — separate corpus

The original benchmark expected IDs refer to 14 full academic documents plus 4 papers, not course rows. Search full unchanged Markdown to reproduce Task 06, with MiniLM native 256-word-piece truncation explicitly disclosed. This is diagnostic source retrieval, not chunking or answer generation. Expected IDs are read **after searching content**, solely for evaluation. Never use expected answers in app output. Summary/comparison/paper analysis remain unimplemented.

In [ ]:
# Instructor benchmark evaluation omitted; inspect your own completed search results.


## Evaluation and failures

Top-1/Top-3/MRR aggregates include only eight single-source direct/lexical/semantic cases. Multi-source, comparison, summary and paper queries report evidence coverage separately. A Top-3 hit is not recall of all required sources. Inspect RET-06 regression and RET-04 improvement; long-file truncation can remove relevant later sections. Do not claim embeddings always win.

In [ ]:
# Instructor benchmark evaluation omitted; inspect your own completed search results.


## End of Day 1

What stayed the same? Cosine, ranking and Top-K. What changed? Representation. Exercise 7 Checkpoint 6 — Semantic Search Works; the original document-chunk checkpoint is still pending. Original semantic-search Checkpoint 4 is now available. Suggested commit: `Build semantic search`.

On Day 2, Transformer concepts and document processing will prepare meaningful retrieval units before generation. No extraction, chunking, RAG or answer generation is implemented here.